In [1]:
"""
Step 01: Download IBKR 1-Minute Bars Into The Staging Folder

Objective:

    - Download SPY 1-minute TRADES bars (regular trading hours) from IBKR, one request per NYSE session, several sessions
      in flight, every session checked minute by minute against the NYSE schedule.
    - Write one file per session into the STAGING folder (store01_rawzone/ibkr_SPY_ohlcv_data_incoming/), never into the raw
      folder that the research pipeline reads. Every attempt is appended to <staging>/download_log.csv.

Prerequisites:

    - IB Gateway (or TWS) running and logged in, API enabled on config.IBKR_PORT_INT (4001 = Gateway live).
    - The network share of nicodesktop reachable (config.DATA_ROOT_PATH_STR).

Outputs:

    - <staging>/ohlcv_data_YYYYMMDD.csv (one per downloaded session), <staging>/download_log.csv.

Next: notebooks/step03_raw_data_check.ipynb (or scripts/merge_staging_into_raw.py) to check the staged sessions and add
them to the raw folder. The same download runs from the command line: python scripts/download_ibkr_ohlcv.py
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
# IMPORT THE INGEST CONFIGURATION
from ingest import config
# IMPORT THE IBKR CLIENT
from ingest.ibkr_client import IbkrApp
# IMPORT THE DOWNLOAD FUNCTIONS
from ingest.ibkr_download import get_download_session_pdf, download_session_pdf
# IMPORT THE MARKET STATUS FUNCTION
from ingest.ibkr_stream import get_market_status_dict
# IMPORT THE MAINTENANCE FUNCTIONS
from ingest.raw_maintenance import get_raw_issue_pdf, merge_staging_into_raw_pdf

In [3]:
# DEFINE THE SESSIONS TO DOWNLOAD (None, None = EVERY CLOSED SESSION AFTER THE LAST ONE IN RAW + STAGING)
date1_str, date2_str = None, None
# DEFINE EXPLICIT DATES INSTEAD OF A RANGE (E.G. ["2009-07-27", "2013-12-23"]; [] = USE THE RANGE)
date_str_list = []
# DEFINE WHETHER SESSIONS ALREADY PRESENT ARE DOWNLOADED AGAIN (INTO STAGING ONLY; RAW IS NEVER REPLACED)
redownload_bool = False
# DISPLAY THE FOLDERS
print(f"Raw folder:\t{config.RAW_OHLCV_PATH_STR}")
print(f"Staging folder:\t{config.STAGING_OHLCV_PATH_STR}")

Raw folder:	//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data/
Staging folder:	//100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data_incoming/


In [4]:
# COLLECT THE MARKET STATUS (REAL NEW YORK TIME)
market_status_dict = get_market_status_dict()
# DISPLAY INFORMATION
pd.Series(market_status_dict)

now_ny_ts                  2026-10-06 13:50:28-04:00
session_today_bool                              True
market_open_bool                                True
first_bar_ts               2026-10-06 09:30:00-04:00
last_bar_ts                2026-10-06 15:59:00-04:00
next_first_bar_ts          2026-10-07 09:30:00-04:00
seconds_until_open_int                             0
seconds_until_close_int                         7772
dtype: object

In [5]:
# CONNECT (REUSE THE CONNECTION IF THIS CELL WAS ALREADY RUN)
try:
    # IF THE APPLICATION EXISTS AND IS CONNECTED
    connected_bool = app.isConnected() or app.connect_app(config.IBKR_HOST_STR, config.IBKR_PORT_INT, config.IBKR_NOTEBOOK_CLIENT_ID_INT)
# IF THE APPLICATION DOES NOT EXIST YET
except NameError:
    # CREATE AND CONNECT IT
    app = IbkrApp()
    connected_bool = app.connect_app(config.IBKR_HOST_STR, config.IBKR_PORT_INT, config.IBKR_NOTEBOOK_CLIENT_ID_INT)

⏳ Connecting to IBKR at 127.0.0.1:4001 (client id 3) ...
✅ Connected to IBKR (ibapi 10.45.1, server version 223)


In [6]:
# SELECT THE SESSIONS TO DOWNLOAD
session_pdf = get_download_session_pdf(date1_str, date2_str, date_str_list, redownload_bool)
# DISPLAY INFORMATION
print(f"Sessions to download: {len(session_pdf):,}")
# PREVIEW DATAFRAME
session_pdf

ℹ️ Last date present in raw + staging: 2026-08-13; starting at 2026-08-14
ℹ️ Not closed yet (skipped): ['2026-10-06']
ℹ️ Already present in raw or staging (skipped): 0 session(s)
Sessions to download: 36


,date,first_bar_ts,last_bar_ts,bar_count_int
0,2026-08-14,2026-08-14 09:30:00-04:00,2026-08-14 15:59:00-04:00,390
1,2026-08-17,2026-08-17 09:30:00-04:00,2026-08-17 15:59:00-04:00,390
2,2026-08-18,2026-08-18 09:30:00-04:00,2026-08-18 15:59:00-04:00,390
3,2026-08-19,2026-08-19 09:30:00-04:00,2026-08-19 15:59:00-04:00,390
4,2026-08-20,2026-08-20 09:30:00-04:00,2026-08-20 15:59:00-04:00,390
5,2026-08-21,2026-08-21 09:30:00-04:00,2026-08-21 15:59:00-04:00,390
6,2026-08-24,2026-08-24 09:30:00-04:00,2026-08-24 15:59:00-04:00,390
7,2026-08-25,2026-08-25 09:30:00-04:00,2026-08-25 15:59:00-04:00,390
8,2026-08-26,2026-08-26 09:30:00-04:00,2026-08-26 15:59:00-04:00,390
9,2026-08-27,2026-08-27 09:30:00-04:00,2026-08-27 15:59:00-04:00,390


In [7]:
# DOWNLOAD THE SESSIONS INTO THE STAGING FOLDER
download_summary_pdf = download_session_pdf(app, session_pdf)
# PREVIEW DATAFRAME
download_summary_pdf

⏳ Downloading 36 session(s) into //100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data_incoming/ (4 in flight, ≥ 0.5 s between requests)
ℹ️ IBKR [2106] HMDS data farm connection is OK:ushmds
✅ [1/36] 2026-08-14 complete (390/390 bars, attempt 1, 5.2 s) -> saved
✅ [2/36] 2026-08-17 complete (390/390 bars, attempt 1, 4.9 s) -> saved
✅ [3/36] 2026-08-18 complete (390/390 bars, attempt 1, 4.4 s) -> saved
✅ [4/36] 2026-08-19 complete (390/390 bars, attempt 1, 4.0 s) -> saved
✅ [5/36] 2026-08-20 complete (390/390 bars, attempt 1, 0.9 s) -> saved
✅ [6/36] 2026-08-21 complete (390/390 bars, attempt 1, 0.7 s) -> saved
✅ [7/36] 2026-08-24 complete (390/390 bars, attempt 1, 0.9 s) -> saved
✅ [8/36] 2026-08-25 complete (390/390 bars, attempt 1, 0.7 s) -> saved
✅ [9/36] 2026-08-26 complete (390/390 bars, attempt 1, 0.9 s) -> saved
✅ [10/36] 2026-08-27 complete (390/390 bars, attempt 1, 0.9 s) -> saved
✅ [11/36] 2026-08-28 complete (390/390 bars, attempt 1, 0.9 s) -> saved
✅ [12/36] 20

,date,status_str,saved_bool,attempt_int,bar_count_int,missing_count_int,file_path_str
0,2026-08-14,complete,True,1,390,0,//100.123.162.2/stock_overflow_data/store01_ra...
1,2026-08-17,complete,True,1,390,0,//100.123.162.2/stock_overflow_data/store01_ra...
2,2026-08-18,complete,True,1,390,0,//100.123.162.2/stock_overflow_data/store01_ra...
3,2026-08-19,complete,True,1,390,0,//100.123.162.2/stock_overflow_data/store01_ra...
4,2026-08-20,complete,True,1,390,0,//100.123.162.2/stock_overflow_data/store01_ra...
5,2026-08-21,complete,True,1,390,0,//100.123.162.2/stock_overflow_data/store01_ra...
6,2026-08-24,complete,True,1,390,0,//100.123.162.2/stock_overflow_data/store01_ra...
7,2026-08-25,complete,True,1,390,0,//100.123.162.2/stock_overflow_data/store01_ra...
8,2026-08-26,complete,True,1,390,0,//100.123.162.2/stock_overflow_data/store01_ra...
9,2026-08-27,complete,True,1,390,0,//100.123.162.2/stock_overflow_data/store01_ra...


In [8]:
# READ THE DOWNLOAD LOG
download_log_pdf = pd.read_csv(config.DOWNLOAD_LOG_FILE_PATH_STR) if not session_pdf.empty else pd.DataFrame()
# PREVIEW THE LAST ATTEMPTS
download_log_pdf.tail(20)

,logged_at_utc,date,attempt_int,status_str,saved_bool,expected_count_int,bar_count_int,missing_count_int,extra_count_int,duplicate_count_int,missing_str,extra_str,error_code_int,error_str,request_seconds_float,end_datetime_str,duration_str,ibapi_version_str
16,2026-10-06T17:50:55+00:00,2026-09-08,1,complete,True,390,390,0,0,0,NaN,NaN,NaN,NaN,0.880,20260908 16:00:00 US/Eastern,23400 S,10.45.1
17,2026-10-06T17:50:55+00:00,2026-09-09,1,complete,True,390,390,0,0,0,NaN,NaN,NaN,NaN,0.650,20260909 16:00:00 US/Eastern,23400 S,10.45.1
18,2026-10-06T17:50:56+00:00,2026-09-10,1,complete,True,390,390,0,0,0,NaN,NaN,NaN,NaN,0.972,20260910 16:00:00 US/Eastern,23400 S,10.45.1
19,2026-10-06T17:50:56+00:00,2026-09-11,1,complete,True,390,390,0,0,0,NaN,NaN,NaN,NaN,0.781,20260911 16:00:00 US/Eastern,23400 S,10.45.1
20,2026-10-06T17:50:57+00:00,2026-09-14,1,complete,True,390,390,0,0,0,NaN,NaN,NaN,NaN,0.878,20260914 16:00:00 US/Eastern,23400 S,10.45.1
21,2026-10-06T17:50:57+00:00,2026-09-15,1,complete,True,390,390,0,0,0,NaN,NaN,NaN,NaN,0.915,20260915 16:00:00 US/Eastern,23400 S,10.45.1
22,2026-10-06T17:50:58+00:00,2026-09-16,1,complete,True,390,390,0,0,0,NaN,NaN,NaN,NaN,0.706,20260916 16:00:00 US/Eastern,23400 S,10.45.1
23,2026-10-06T17:50:59+00:00,2026-09-17,1,complete,True,390,390,0,0,0,NaN,NaN,NaN,NaN,0.930,20260917 16:00:00 US/Eastern,23400 S,10.45.1
24,2026-10-06T17:50:59+00:00,2026-09-18,1,complete,True,390,390,0,0,0,NaN,NaN,NaN,NaN,0.960,20260918 16:00:00 US/Eastern,23400 S,10.45.1
25,2026-10-06T17:50:59+00:00,2026-09-21,1,complete,True,390,390,0,0,0,NaN,NaN,NaN,NaN,0.788,20260921 16:00:00 US/Eastern,23400 S,10.45.1


In [9]:
# CHECK THE STAGED SESSIONS TOGETHER WITH THE RAW FOLDER (READ ONLY)
issue_pdf = get_raw_issue_pdf([config.RAW_OHLCV_PATH_STR, config.STAGING_OHLCV_PATH_STR], session_pdf["date"].min() if not session_pdf.empty else None)
# PREVIEW DATAFRAME
issue_pdf

ℹ️ Skipped non-OHLCV files in //100.123.162.2/stock_overflow_data/store01_rawzone/ibkr_ohlcv_data_incoming/: ['download_log.csv']
Rows: 2,125,831 | sessions expected: 36 (2026-08-14 -> 2026-10-05) | problem dates: 0
  missing_session (0): []
  missing_minutes (0): []
  extra_minutes (0): []
  duplicate_rows (0): []
  no_session (0): []


,date,issue_str,expected_count_int,bar_count_int,unique_count_int,missing_count_int,missing_str,extra_count_int,extra_str,duplicate_count_int


In [10]:
# DRY RUN OF THE MERGE INTO THE RAW FOLDER (NOTHING IS WRITTEN; USE step03 OR THE SCRIPT WITH --apply TO ADD THE FILES)
merge_check_pdf = merge_staging_into_raw_pdf(apply_bool_in=False)
# PREVIEW DATAFRAME
merge_check_pdf

🔎 [DRY RUN] Staging -> raw: add: 36


,date,check_status_str,bar_count_int,missing_count_int,sanity_issue_count_int,bad_tick_count_int,in_raw_bool,decision_str,file_name_str
0,2026-08-14,complete,390,0,0,0,False,add,ohlcv_data_20260814.csv
1,2026-08-17,complete,390,0,0,0,False,add,ohlcv_data_20260817.csv
2,2026-08-18,complete,390,0,0,0,False,add,ohlcv_data_20260818.csv
3,2026-08-19,complete,390,0,0,0,False,add,ohlcv_data_20260819.csv
4,2026-08-20,complete,390,0,0,0,False,add,ohlcv_data_20260820.csv
5,2026-08-21,complete,390,0,0,0,False,add,ohlcv_data_20260821.csv
6,2026-08-24,complete,390,0,0,0,False,add,ohlcv_data_20260824.csv
7,2026-08-25,complete,390,0,0,0,False,add,ohlcv_data_20260825.csv
8,2026-08-26,complete,390,0,0,0,False,add,ohlcv_data_20260826.csv
9,2026-08-27,complete,390,0,0,0,False,add,ohlcv_data_20260827.csv


In [11]:
# DISCONNECT
app.disconnect_app()

✅ Disconnecting from IBKR ...
